In [0]:
import os
from pyspark.sql import functions as F
from pathlib import Path
source = '/Workspace/Shared/HANDSON SANDBOX/downloads'
subfolders = [Path(f.path) for f in os.scandir(source) if f.is_dir()]
latest_folder = max(subfolders, key=lambda f: f.stat().st_ctime)

# print(latest_folder)
for i in os.listdir(latest_folder):
    if i.endswith('_all.csv') and i.startswith('Northfield_EMR_user'):
        df = spark.read.csv('file:' + str(latest_folder) + '/' + i, header=True, inferSchema=True)
        df = df.withColumn('crdt_dt', F.current_timestamp()).withColumn('src', F.lit(i[15:-8]))
        # print(i)
        df.write.mode('append').saveAsTable(f'us_scdb_phi_dev.comm_bronze.{i[15:-24]}')

In [0]:
import re
tables = spark.sql('show tables in us_scdb_phi_dev.comm_bronze').collect()
pattern = re.compile(r'^user_\d{1,2}$')
for row in tables:
    table_name = row['tableName']
    ls1 = ['patient_name', 'dob','mrn', 'username']
    if not pattern.match(table_name): continue
    for j in ls1:
        spark.sql(f"alter table us_scdb_phi_dev.comm_bronze.{table_name} alter column {j} SET TAGS ('phi' = '')")
    # print(table_name)

In [0]:
# spark.sql("""select * from us_scdb_phi_dev.information_schema.columns where table_schema='comm_bronze' and column_name in ('patient_name', 'dob','mrn', 'username') """).collect()
